### Prompt Templates


In [7]:
import time

from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter

load_dotenv()

True

In [8]:
model = ChatOpenRouter(model="nvidia/nemotron-3-ultra-550b-a55b:free")

#### 1. Problem

Summarise a customer email, built by hand:

In [9]:
email = "The parcel arrived three days late.The box was crushed. It seems to be empty."

prompt_text = "Summarize this email in one sentence:" + email

model.invoke(prompt_text).text

'The parcel arrived three days late with a crushed, apparently empty box.'

#### 2. Write the prompt once with ChatPrmptTemplate

A template is the prompt with named holes in it.

In [13]:
from langchain_core.prompts import ChatPromptTemplate

summarizer = ChatPromptTemplate.from_messages([
    ("system", "You summarise customer emails for a support team."),
    ("user", "Summarise this email in one sentence:\n\n{email}"),
])

{email} is the hole, nothing filled in yet, no model called.

("system", ...) and ("user", ...) are shorthand for the SystemMessage/HumanMessage

#### 3. Fill the template and inspect the messages

In [15]:
summarizer

ChatPromptTemplate(input_variables=['email'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You summarise customer emails for a support team.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['email'], input_types={}, partial_variables={}, template='Summarise this email in one sentence:\n\n{email}'), additional_kwargs={})])

In [17]:
filled = summarizer.invoke({"email": email})
filled.messages

[SystemMessage(content='You summarise customer emails for a support team.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Summarise this email in one sentence:\n\nThe parcel arrived three days late.The box was crushed. It seems to be empty.', additional_kwargs={}, response_metadata={})]

A template is not a new kind of thing. It's a factory that produces a message list, and you can always look at what it produced.

In [18]:
model.invoke(filled).text

'The customer received a parcel three days late with a crushed box that appears to be empty.'

In [19]:
from langchain.messages import HumanMessage, SystemMessage

model.invoke([SystemMessage(content='You summarise customer emails for a support team.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Summarise this email in one sentence:\n\nThe parcel arrived three days late.The box was crushed. It seems to be empty.', additional_kwargs={}, response_metadata={})])

AIMessage(content='The customer reports that their parcel arrived three days late with a crushed box that appears to be empty.', additional_kwargs={'reasoning_content': 'The user wants a one-sentence summary of a short customer email.\nKey points: parcel 3 days late, box crushed, appears empty.\nI need to combine these into a single sentence.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user wants a one-sentence summary of a short customer email.\nKey points: parcel 3 days late, box crushed, appears empty.\nI need to combine these into a single sentence.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1792590152-WyQ1LF5izn1eoZrDasK3', 'created': 1792590152, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_co

#### 4. input_variables: what the template expects

In [20]:
summarizer.input_variables

['email']

Useful when a prompt has grown and you've lost track of what it expects.

#### 5. A template with two variables

Templates get useful when there is more than one variable.

In [21]:
replier = ChatPromptTemplate.from_messages([
    ("system", """You write replies for a customer support team.
Write in a {tone} tone. Keep it under 60 words.
Never promise a refund."""),
    ("user", "Reply to this email: {email}"),
])

In [24]:
replier.invoke({"tone": "warm", "email": email})

ChatPromptValue(messages=[SystemMessage(content='You write replies for a customer support team.\nWrite in a warm tone. Keep it under 60 words.\nNever promise a refund.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Reply to this email: The parcel arrived three days late.The box was crushed. It seems to be empty.', additional_kwargs={}, response_metadata={})])

In [ ]:
time.sleep(4)

model.invoke(replier.invoke({"tone": "warm", "email": email})).text

'I’m so sorry your parcel arrived late and damaged—and seemingly empty. That’s certainly not the experience we want for you. I’ve opened an investigation with the carrier right away and will update you within 24 hours. We’ll make sure this gets resolved for you quickly.'

In [26]:
time.sleep(4)

model.invoke(replier.invoke({"tone": "formal", "email": email})).text

'Dear Customer,\n\nWe sincerely apologize for the delay and the condition of your parcel. This is not the standard we uphold. We have immediately escalated this to our logistics partner for a full investigation. A member of our resolution team will contact you within 24 hours to discuss the next steps regarding your missing items.\n\nSincerely,\nCustomer Support Team'

One template, two tones, no string gluing.

Never promise a refund lives in the system half. Policies belong in the template, written once, not retyped at every call site.

#### 6. A missing variable raises KeyError

In [ ]:
# replier.invoke({"email": email})

In [27]:
try:
    replier.invoke({"email": email})
except KeyError as problem:
    print(problem)

"Input to ChatPromptTemplate is missing variables {'tone'}.  Expected: ['email', 'tone'] Received: ['email']\nNote: if you intended {tone} to be part of the string and not a variable, please escape it with double curly braces like: '{{tone}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "


It refused rather than sending a broken prompt. A hand-built string would have quietly produced in a  tone and cost you an hour of confusion. Loud failures are cheaper than quiet ones.

#### 7. Literal braces: why JSON in a prompt breaks

Braces mean "hole". What happens when the prompt genuinely needs one, like JSON?

#### 

In [28]:
trap = ChatPromptTemplate.from_messages([
    ("system", 'Reply as JSON like {"airline": "Zephyr}'),
    ("user", "{complaint}"),
])

trap.input_variables

['"airline"', 'complaint']


Two variables now, one of them "airline". The template read the JSON example as a hole to fill.

In [29]:
try:
    trap.invoke({"complaint": "My flight was cancelled."})
except KeyError as problem:
    print(problem)


'Input to ChatPromptTemplate is missing variables {\'"airline"\'}.  Expected: [\'"airline"\', \'complaint\'] Received: [\'complaint\']\nNote: if you intended {"airline"} to be part of the string and not a variable, please escape it with double curly braces like: \'{{"airline"}}\'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT '



The fix is to double every literal brace.

In [30]:
fixed = ChatPromptTemplate.from_messages([
    ("system", 'Reply as JSON like {{"airline": "Zephyr"}}'),
    ("user", "{complaint}"),
])

fixed.input_variables

['complaint']

In [31]:
fixed.invoke({"complaint": "My flight was cancelled."})

ChatPromptValue(messages=[SystemMessage(content='Reply as JSON like {"airline": "Zephyr"}', additional_kwargs={}, response_metadata={}), HumanMessage(content='My flight was cancelled.', additional_kwargs={}, response_metadata={})])

#### 8. Pinning a value with partial

partial fills one hole now and hands back a template with the rest still open.

In [32]:
warm_replier = replier.partial(tone="warm")

warm_replier

ChatPromptTemplate(input_variables=['email'], input_types={}, partial_variables={'tone': 'warm'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['tone'], input_types={}, partial_variables={}, template='You write replies for a customer support team.\nWrite in a {tone} tone. Keep it under 60 words.\nNever promise a refund.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['email'], input_types={}, partial_variables={}, template='Reply to this email: {email}'), additional_kwargs={})])

In [33]:
time.sleep(4)

model.invoke(warm_replier.invoke({"email":email})).text

"I'm so sorry your order arrived late and damaged — that's not the experience we want for you. I've opened an investigation with the carrier and our fulfillment team to locate your items. We'll email you within 24 hours with an update and next steps. Thank you for your patience."

tone is gone from what you must supply: one template, two configurations, simpler calling code.

This is how a team gets a prompt pre-configured for its use case, without copying it.

#### 9. MessagesPlaceholder for chat history

A hole does not have to be a string. MessagesPlaceholder reserves a slot for a list of messages.

In [34]:
from langchain_core.prompts import MessagesPlaceholder

chat_prompt = ChatPromptTemplate.from_messages([
    ("system","You ara a concise travel assistant."),
    MessagesPlaceholder("history"),
    ("user","{question}"),
])

In [35]:
chat_prompt

ChatPromptTemplate(input_variables=['history', 'question'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchai

In [36]:
from langchain.messages import HumanMessage, AIMessage

past = [
    HumanMessage("I am flying to Delhi on Friday."),
    AIMessage("Noted, Delhi on Friday."),
]

built = chat_prompt.invoke({"history":past, "question":"What day did I say?"})

built.messages

[SystemMessage(content='You ara a concise travel assistant.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I am flying to Delhi on Friday.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Noted, Delhi on Friday.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What day did I say?', additional_kwargs={}, response_metadata={})]

In [37]:
model.invoke(built).text

'You said **Friday**.'

This is the exact shape of a chatbot prompt, fixed instructions in the system message, never tangled.

#### 10. Run one template over many inputs

In [39]:
emails = [
    "The parcel arrived three days late and the box was crushed.",
    "I was charged twice for the same order.",
    "Your app logged me out and will not let me back in.",
]

for e in emails:
    time.sleep(4)
    reply = model.invoke(warm_replier.invoke({"email": e}))
    print(reply.text)
    print("-" * 60)

Hi there,

I’m so sorry your order arrived late and damaged—that’s not the experience we want for you. I’ve already flagged this with our shipping partner to prevent it from happening again. Please reply with a photo of the box and the items inside, and I’ll get a replacement sent out right away.
------------------------------------------------------------
Hi there,

Thanks for reaching out! I’m sorry to see this duplicate charge. I’ve escalated this to our billing team for immediate review. They’ll investigate the transaction and correct the error shortly. You’ll hear back via email once it’s resolved.

Thanks for your patience!

Best,
The Support Team
------------------------------------------------------------
Hi there,

I’m sorry you’re locked out—that’s frustrating. Let’s get you back in. First, try force-closing the app and restarting your device. If that doesn’t work, please ensure you’re on the latest version or try a password reset via the login screen.

Still stuck? Reply her


Three replies from one template. Change the policy once and all three change.

### 11. from_template, for prompts with no system message

In [41]:
quick = ChatPromptTemplate.from_template("TRanslate to French: {text}")

quick.invoke({"text": "good morning"}).messages

[HumanMessage(content='TRanslate to French: good morning', additional_kwargs={}, response_metadata={})]

In [42]:
quick

ChatPromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='TRanslate to French: {text}'), additional_kwargs={})])

One HumanMessage, no system message. Convenient for small jobs; reach for from_messages as soon as there's an instruction worth stating.